# 02 — Set up the database

Spin up Neo4j, apply the clg schema (constraints + indexes + vector index + jurisdiction seeds), and demonstrate the **removability invariant** (Phase 11) — flipping `ENABLED_JURISDICTIONS` silences a jurisdiction without deleting data.

## Prereqs
- Docker running (we use the project's `docker-compose.yml`).
- `.env` with Neo4j credentials (defaults work for a local dev install).

## What the schema applies
- 8 uniqueness constraints (one per node label).
- 6 lookup indexes (case+jurisdiction, decision_date, provision valid_from / section_path / instrument_id / composite).
- 1 vector index on `Chunk.embedding` keyed by `Settings.embedding_dim`.
- 5 `Jurisdiction` seed nodes (US, EW, UK, EU, DK) — **only the ones in `ENABLED_JURISDICTIONS`**.

In [1]:
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

def run(cmd: str, check: bool = True) -> str:
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout)
    if r.stderr.strip():
        print('--- stderr ---'); print(r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')
    return r.stdout

## Step 1 — start Neo4j

In [2]:
run('docker compose up -d neo4j')

# Wait until the bolt port responds. The first start takes ~10s.
import time
for _ in range(30):
    try:
        import httpx
        r = httpx.get('http://localhost:7474', timeout=2.0)
        if r.status_code == 200:
            print('neo4j reachable')
            break
    except Exception:
        time.sleep(1)

$ docker compose up -d neo4j

--- stderr ---
 Container crimellm-neo4j  Running

neo4j reachable


## Step 2 — apply schema

`apply_schema()` is **idempotent** and **non-destructive**: re-running it after a config change is safe. The `jurisdictions_skipped` counter tells you how many seeds were intentionally not created on this run.

In [3]:
from crimellm.clg.config import get_settings
from crimellm.clg.graph import apply_schema, get_store

# Force the default settings load.
get_settings.cache_clear()
settings = get_settings()
print('enabled jurisdictions:', settings.enabled_jurisdictions)
print('embedding model     :', settings.embedding_model)
print('embedding dim       :', settings.embedding_dim)

store = get_store()
store.verify()
counts = apply_schema(store)
print('\napply_schema counts:', counts)

enabled jurisdictions: ['US', 'EW', 'UK', 'EU', 'DK']
embedding model     : sentence-transformers/all-roberta-large-v1
embedding dim       : 1024

apply_schema counts: {'constraints': 8, 'indexes': 7, 'vector_index': 1, 'jurisdictions': 5, 'jurisdictions_skipped': 0}


## Step 3 — inspect schema + node counts via the CLI

`clg graph status` returns the live constraint / index list + a per-label node count.

In [4]:
run('uv run clg graph status')

$ uv run clg graph status
{
  "schema": {
    "constraints": [
      "case_id",
      "chunk_id",
      "concept_id",
      "court_id",
      "instrument_id",
      "judge_id",
      "jurisdiction_code",
      "provision_id"
    ],
    "indexes": [
      "case_decision_date",
      "case_id",
      "case_jurisdiction",
      "chunk_embedding",
      "chunk_id",
      "concept_id",
      "court_id",
      "index_343aff4e",
      "index_f7700477",
      "instrument_id",
      "instrument_jurisdiction",
      "judge_id",
      "jurisdiction_code",
      "provision_id",
      "provision_instrument",
      "provision_section_lookup",
      "provision_section_path",
      "provision_valid_from"
    ]
  },
  "node_counts": [
    {
      "label": "Case",
      "n": 2
    },
    {
      "label": "Instrument",
      "n": 1
    },
    {
      "label": "Jurisdiction",
      "n": 5
    },
    {
      "label": "Provision",
      "n": 6
    }
  ]
}



'{\n  "schema": {\n    "constraints": [\n      "case_id",\n      "chunk_id",\n      "concept_id",\n      "court_id",\n      "instrument_id",\n      "judge_id",\n      "jurisdiction_code",\n      "provision_id"\n    ],\n    "indexes": [\n      "case_decision_date",\n      "case_id",\n      "case_jurisdiction",\n      "chunk_embedding",\n      "chunk_id",\n      "concept_id",\n      "court_id",\n      "index_343aff4e",\n      "index_f7700477",\n      "instrument_id",\n      "instrument_jurisdiction",\n      "judge_id",\n      "jurisdiction_code",\n      "provision_id",\n      "provision_instrument",\n      "provision_section_lookup",\n      "provision_section_path",\n      "provision_valid_from"\n    ]\n  },\n  "node_counts": [\n    {\n      "label": "Case",\n      "n": 2\n    },\n    {\n      "label": "Instrument",\n      "n": 1\n    },\n    {\n      "label": "Jurisdiction",\n      "n": 5\n    },\n    {\n      "label": "Provision",\n      "n": 6\n    }\n  ]\n}\n'

## Step 4 — direct Cypher

Sometimes you want to bypass the CLI and inspect the graph directly.

In [5]:
rows = store.run(
    'MATCH (j:Jurisdiction) '
    'OPTIONAL MATCH (j)<-[:IN_JURISDICTION]-(i:Instrument) '
    'OPTIONAL MATCH (j)<-[:IN_JURISDICTION]-(c:Case) '
    'RETURN j.code AS code, j.name AS name, count(DISTINCT i) AS instruments, count(DISTINCT c) AS cases '
    'ORDER BY j.code'
)
for r in rows:
    print(f"{r['code']:<3} {r['name']:<22} instruments={r['instruments']:>4}  cases={r['cases']:>6}")

DK  Denmark                instruments=   0  cases=     0
EU  European Union         instruments=   0  cases=     0
EW  England & Wales        instruments=   0  cases=     0
UK  United Kingdom         instruments=   1  cases=     0
US  United States          instruments=   0  cases=     2


## Step 5 — Removability invariant (Phase 11)

Flip `ENABLED_JURISDICTIONS` → `apply_schema()` skips the disabled codes but **never deletes** existing seeds or data.

In [6]:
# Simulate a DK-firm config: only EU + DK active.
os.environ['ENABLED_JURISDICTIONS'] = 'EU,DK'
get_settings.cache_clear()
store.settings = get_settings()

counts = apply_schema(store)
print('counts when only EU + DK enabled:', counts)
# jurisdictions_skipped > 0 means US/EW/UK seeds were left alone.

# Existing seeds for disabled jurisdictions persist.
rows = store.run('MATCH (j:Jurisdiction) RETURN j.code AS code ORDER BY code')
print('seeds in graph:', [r['code'] for r in rows])

counts when only EU + DK enabled: {'constraints': 8, 'indexes': 7, 'vector_index': 1, 'jurisdictions': 2, 'jurisdictions_skipped': 3}
seeds in graph: ['DK', 'EU', 'EW', 'UK', 'US']


In [ ]:
# Restore the full enabled set.
os.environ['ENABLED_JURISDICTIONS'] = 'US,EW,UK,EU,DK'
get_settings.cache_clear()
store.settings = get_settings()
print('restored:', store.settings.enabled_jurisdictions)

## Step 6 — when to rebuild the vector index

`Chunk.embedding` is keyed by dim. If you switch embedder to a different-dim model (e.g. `BAAI/bge-m3` 1024 → `Qwen/Qwen3-Embedding-8B` 4096), rebuild the index. Same-dim swaps don't need this (use `clg embed-rebuild` per jurisdiction instead).

**Destructive flag warning:** `--drop-chunks --yes` deletes every `Chunk` node. Re-embed (notebook 03) to repopulate.

In [ ]:
# Demo only — uncomment when actually swapping embedder dims.
# run('uv run clg graph rebuild-vector-index --dim 4096 --drop-chunks --yes')
print('skipped — uncomment only when changing embedding_dim')

## Wipe (destructive!)

If you need to start over from an empty graph:

In [ ]:
# run('uv run clg graph wipe --yes')   # nuke all nodes + relationships
print('skipped — destructive')

## Next: notebook 03 — parse + load + embed

[`03_load_and_embed.ipynb`](03_load_and_embed.ipynb) runs the parse → load → IMPLEMENTS → chunk → embed pipeline against the fixtures (or whatever you downloaded in notebook 01). After that, the graph is queryable end-to-end.